<a href="https://colab.research.google.com/github/Jumpr15/pytorch-work/blob/main/CartPole_Gym_REINFORCE_revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install flappy-bird-gymnasium gymnasium[other]

In [ ]:
import torch.distributions as distributions
import torch.nn as nn
import torch

class Policy:
  def __init__(self, model, optimizer):
    self.model = model
    self.optimizer = optimizer

  def sample_action(self, obs):
    probs = self.model(obs)
    distri = distributions.Categorical(probs=probs)
    action = distri.sample()
    return action.item()

  def discount_rewards(self, reward_tensor, discount_factor=0.99):
    discount_list = []
    reward_sum = 0

    for reward in reversed(reward_tensor):
        reward_sum = reward + (reward_sum * discount_factor)
        discount_list.append(reward_sum)

    discounted_tensor = torch.flip(torch.tensor(discount_list), dims=[0])
    final_rewards = (discounted_tensor - discounted_tensor.mean()) / (discounted_tensor.std() + 1e-8)
    return final_rewards

  def calculate_loss(self, train_states, train_actions, train_rewards):
    train_states = torch.tensor(train_states, dtype=torch.float32)
    train_actions = torch.tensor(train_actions, dtype=torch.float32)
    train_rewards = torch.tensor(train_rewards, dtype=torch.float32)

    discounted_rewards = self.discount_rewards(train_rewards)

    probs = self.model(train_states)
    distri = distributions.Categorical(probs=probs)
    loss = -distri.log_prob(train_actions) * discounted_rewards
    return loss.mean()

  def optimize_policy(self, train_states, train_actions, train_rewards):
    loss = self.calculate_loss(train_states, train_actions, train_rewards)

    self.optimizer.zero_grad()
    loss.backward()

    nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=0.6)
    self.optimizer.step()

    return loss.item()

In [ ]:
import torch.nn as nn
import torch.optim as optim

in_dims = 4
h_dims = 16
out_dims = 2
model = nn.Sequential(
    nn.Linear(in_dims, h_dims),
    nn.ReLU(),
    nn.Linear(h_dims, h_dims),
    nn.ReLU(),
    nn.Linear(h_dims, out_dims),
    nn.Softmax(dim=-1)
)

lr = 1e-4
optimizer = optim.Adam(
    model.parameters(),
    lr=lr
)

policy = Policy(model, optimizer)

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

env = RecordVideo(gym.make("CartPole-v1", render_mode="rgb_array"), video_folder='./video-output-3')

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content/video-output-3 folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


In [ ]:
def train_epoch(env):
  obs, _ = env.reset()

  train_states = []
  train_actions = []
  train_rewards = []

  while True:
    obs_tensor = (torch.from_numpy(obs)).to(torch.float32)
    action = policy.sample_action(obs_tensor)

    train_states.append(obs)
    train_actions.append(action)

    obs, reward, terminated, _, info = env.step(action)

    train_rewards.append(reward)

    if terminated:
      break

  env.close()
  return train_states, train_actions, train_rewards

In [ ]:
torch.manual_seed(67)
episodes = 10000
for _ in range(episodes):
  train_states, train_actions, train_rewards = train_epoch(env)
  loss = policy.optimize_policy(train_states, train_actions, train_rewards)
  print(loss)

-0.009765470400452614
-0.006516871973872185
-0.006853173486888409
-0.01404189970344305
-0.029245654121041298
-0.007542056031525135
-0.017962642014026642
-0.028160834684967995
-0.00910102017223835
-0.01532133761793375
-0.01105667743831873
-0.01853829436004162
-0.008174827322363853
-0.017922325059771538
0.02050335891544819
-0.0158977247774601
-0.025919558480381966
-0.022044802084565163
-3.5434601159067824e-05
-0.02334870956838131
-0.015167880803346634
0.0004976534401066601
-0.029143793508410454
-0.020674465224146843
-0.003831725800409913
-0.019103344529867172
-0.010393911972641945
-0.03528565168380737
-0.03305728733539581
-0.026561636477708817
-0.013753760606050491
-0.01557310950011015
-0.04963406175374985
-0.016650963574647903
-0.04070849344134331
-0.0009432973456569016
-0.05301593616604805
0.00727127306163311
-0.0030382825061678886
-0.026388296857476234
-0.02441760152578354
-0.032868120819330215
-0.007490936201065779
0.004679408855736256
-0.003761507570743561
-0.03331592679023743
-0.01